# MV00 DDP-B16 — Continuous multi-epoch Kaggle session

Uses the cross-session-tested **G4 epoch-1 checkpoint** as the starting point; the audited scientific source remains pinned to `f440d329d4544ec08ad63c1972bcf9a8b8a7eed8`. One Save Version run advances as many complete epochs as fit safely within its session, keeping only the most recent validated G4 package in output. The runner stops only at an epoch boundary; no B-TEST is accessed. **Candidate/not officially accepted until live session output is inspected.**

**Next session:** attach the preceding notebook-version output as input and edit just `PREVIOUS_OUTPUT_DATASET` and `PREVIOUS_MANIFEST_SHA256` in the code cell. Do not alter the fixed recipe or Git SHA. Use **T4 × 2** and **Save Version → Save & Run All**. Keep internet off; the exact source is restored from the SHA-pinned bundle already attached.


In [ ]:
WORKER_CODE = "\"\"\"Pinned-source MV00 DDP-B16 multi-epoch Kaggle continuation (nonofficial).\n\nThe scientific source stays at f440d329d4544ec08ad63c1972bcf9a8b8a7eed8.\nReal torchrun ranks separately authenticate R0, source stage and external G4;\nthe fixed native trainer handles all optimizer, augmentation and B-VAL math.\nOnly the verified epoch-G4 boundary may stop a session. B-TEST is sealed.\n\"\"\"\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport os\nimport shutil\nimport sys\nimport time\nfrom datetime import timedelta\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.distributed as dist\n\nROOT = Path(\"/kaggle/working/MV00_B16_CONTINUOUS\")\nREPO = ROOT / \"repo\"\nG4 = ROOT / \"g4\"\nOUT = Path(\"/kaggle/working/MV00_B16_CONTINUOUS_OUTPUT\")\nPLAN = ROOT / \"plan.json\"\nEXTERNAL_PIN = os.environ[\"DDP_B16_PREVIOUS_MANIFEST_SHA256\"]\nBUDGET_HOURS = float(os.environ.get(\"DDP_B16_SESSION_HOURS\", \"9.0\"))\nsys.dont_write_bytecode = True\nsys.path.insert(0, str(REPO))\n\nfrom research.r9_training.ddp_b16.ddp_worker import review_worker_inputs, _unique_json\nfrom research.r9_training.ddp_b16.ddp_verified_admission import verify_staged_admission\nfrom research.r9_training.ddp_b16.ddp_rank_admission import require_review_consensus\nfrom research.r9_training.ddp_b16.ddp_native_binding import make_native_binding\nfrom research.r9_training.ddp_b16.ddp_native_binding_consensus import require_native_binding_consensus\nfrom research.r9_training.ddp_b16.ddp_native_bootstrap_candidate import verify_two_t4_runtime, load_pinned_native_class\nfrom research.r9_training.ddp_b16.ddp_native_state import SCHEMA as STATE_SCHEMA\nfrom research.r9_training.ddp_b16 import ddp_native_g4_hook as hook\n\n\nclass EpochBoundaryStop(Exception):\n    pass\n\n\ndef require(value, code):\n    if not value:\n        raise RuntimeError(\"MV00_MULTI_EPOCH_\" + code)\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\"rb\") as f:\n        for part in iter(lambda: f.read(4 * 1024 * 1024), b\"\"):\n            h.update(part)\n    return h.hexdigest()\n\n\ndef write_exclusive_json(path, obj):\n    with Path(path).open(\"x\", encoding=\"utf-8\") as f:\n        json.dump(obj, f, sort_keys=True, indent=2, allow_nan=False)\n        f.write(\"\\n\")\n\n\ndef persist_epoch(trainer, result, identity, epoch, previous_sha):\n    \"\"\"Rank 0: SHA-verify all six members and rotate one resumable output.\"\"\"\n    step = epoch + 1\n    filename = f'{identity[\"run_id\"]}_epoch{step:03d}_two_rank_manifest.json'\n    manifest_path = G4 / filename\n    require(manifest_path.is_file() and\n            sha256(manifest_path) == result[\"manifest_sha256\"], \"MANIFEST_HASH\")\n    manifest = json.loads(manifest_path.read_text(\"utf-8\"))\n    require(manifest.get(\"complete\") is True and\n            manifest.get(\"identity\") == identity and\n            manifest.get(\"cursor\", {}).get(\"next_epoch\") == step and\n            manifest[\"cursor\"][\"completed_epoch\"] == epoch and\n            manifest.get(\"b_test_access\") == \"NONE\" and\n            set(manifest[\"ranks\"]) == {\"0\", \"1\"}, \"MANIFEST_CONTENT\")\n    files = {filename: result[\"manifest_sha256\"],\n             \"best.pt\": result[\"selected_best_sha256\"]}\n    for rank in (0, 1):\n        member = manifest[\"ranks\"][str(rank)]\n        files[member[\"filename\"]] = member[\"sha256\"]\n        files[member[\"receipt_filename\"]] = member[\"receipt_sha256\"]\n    require(len(files) == 6, \"SIX_G4_MEMBERS\")\n    temporary = OUT / f\".next_{step:03d}\"\n    require(not temporary.exists(), \"TEMP_EXISTS\")\n    temporary.mkdir()\n    for name, digest in files.items():\n        require(Path(name).name == name, \"PATH_TRAVERSAL\")\n        origin = Path(trainer.best) if name == \"best.pt\" else G4 / name\n        require(origin.is_file() and not origin.is_symlink() and\n                sha256(origin) == digest, \"G4_SOURCE_SHA\")\n        shutil.copyfile(origin, temporary / name)\n        require(sha256(temporary / name) == digest, \"G4_EXPORT_SHA\")\n    receipt = {\n        \"schema\": \"MV00_DDP_B16_CONTINUOUS_OUTPUT_V1\",\n        \"status\": \"MV00_RESUMABLE_EPOCH_G4_CANDIDATE\",\n        \"identity\": identity, \"run_id\": identity[\"run_id\"],\n        \"source_release_sha\": identity[\"ddp_release_sha\"],\n        \"completed_epoch\": step, \"next_epoch\": step,\n        \"optimizer_updates\": manifest[\"cursor\"][\"optimizer_updates\"],\n        \"manifest_sha256\": result[\"manifest_sha256\"],\n        \"previous_manifest_sha256\": previous_sha,\n        \"selected_best_sha256\": result[\"selected_best_sha256\"],\n        \"files_sha256\": files, \"b_test_access\": \"NONE\",\n        \"official_training_authorized\": False,\n        \"offsession_restoration_verified\": False,\n    }\n    write_exclusive_json(temporary / \"ACCEPTANCE.json\", receipt)\n    latest, previous = OUT / \"g4\", OUT / \".previous_g4\"\n    require(not previous.exists(), \"PREVIOUS_ROTATION_UNFINISHED\")\n    if latest.exists():\n        latest.rename(previous)\n    temporary.rename(latest)\n    if previous.exists():\n        shutil.rmtree(previous)\n    # Prune only generated work-state copies; the newest exported G4 is intact.\n    for path in G4.iterdir():\n        if path.is_file() and path.name not in files and path.name != \"best.pt\":\n            path.unlink()\n    return receipt\n\n\ndef persist_terminal(trainer, identity):\n    \"\"\"Executed only after native selected-BEST final B-VAL succeeds.\"\"\"\n    require(trainer.epoch == 99 and\n            trainer.validator.seen == 3049 and\n            trainer.validator._d4_fixed_seen == 3049, \"TERMINAL_BVAL\")\n    metrics = {str(k): float(v) for k, v in trainer.metrics.items()}\n    require(bool(metrics) and all(math.isfinite(x) for x in metrics.values()),\n            \"TERMINAL_METRICS\")\n    terminal = OUT / \"terminal\"\n    require(not terminal.exists(), \"TERMINAL_OUTPUT_EXISTS\")\n    terminal.mkdir()\n    files = {}\n    for name, path in ((\"best.pt\", Path(trainer.best)),\n                       (\"last.pt\", Path(trainer.last))):\n        require(path.is_file() and not path.is_symlink(),\n                \"TERMINAL_CHECKPOINT\")\n        files[name] = sha256(path)\n        shutil.copyfile(path, terminal / name)\n        require(sha256(terminal / name) == files[name], \"TERMINAL_SHA\")\n    receipt = {\n        \"schema\": \"MV00_DDP_B16_TERMINAL_V1\",\n        \"status\": \"EPOCH100_FINISHED_SELECTED_BEST_BVAL_CANDIDATE\",\n        \"identity\": identity, \"completed_epochs\": 100,\n        \"metrics\": metrics, \"operational_val_images\": 3049,\n        \"files_sha256\": files, \"b_test_access\": \"NONE\",\n        \"official_training_authorized\": False,\n    }\n    write_exclusive_json(terminal / \"TERMINAL.json\", receipt)\n    return receipt\n\n\ndef main():\n    rank, local, world = (int(os.environ[k])\n                          for k in (\"RANK\", \"LOCAL_RANK\", \"WORLD_SIZE\"))\n    require(os.environ.get(\"CUDA_VISIBLE_DEVICES\") == \"0,1\" and\n            (world, rank, local) in ((2, 0, 0), (2, 1, 1)), \"RANKS\")\n    require(0.5 <= BUDGET_HOURS <= 10.5, \"SESSION_BUDGET\")\n    plan = _unique_json(PLAN)\n    manifests = list(G4.glob(\"*_two_rank_manifest.json\"))\n    require(len(manifests) == 1 and\n            sha256(manifests[0]) == EXTERNAL_PIN, \"EXTERNAL_MANIFEST_PIN\")\n    manifest = json.loads(manifests[0].read_text(\"utf-8\"))\n    identity, cursor = manifest[\"identity\"], manifest[\"cursor\"]\n    require(identity[\"arm\"] == \"MV00\" and\n            identity[\"run_id\"] == plan[\"run_id\"] and\n            identity[\"ddp_release_sha\"] == plan[\"expected_git_head\"] and\n            1 <= cursor[\"next_epoch\"] < 100, \"RESUME_IDENTITY\")\n    require(G4.is_dir() and OUT.is_dir() and not (OUT / \"g4\").exists(),\n            \"NEW_OUTPUT_REQUIRED\")\n\n    dist.init_process_group(\"gloo\", init_method=\"env://\",\n                            rank=rank, world_size=2,\n                            timeout=timedelta(seconds=600))\n    try:\n        review = review_worker_inputs(plan, stage_source=False)\n        admission = verify_staged_admission(plan, review)\n        require_review_consensus(admission)\n        config = _unique_json(Path(plan[\"r0_input_config\"]))\n        binding = make_native_binding(\n            plan, admission, config, project=str(ROOT / \"native_project\"))\n        require_native_binding_consensus(binding)\n        require(binding[\"identity\"] == identity and\n                binding[\"overrides\"][\"epochs\"] == 100 and\n                binding[\"overrides\"][\"batch\"] == 16 and\n                binding[\"overrides\"][\"resume\"] is False, \"FROZEN_RECIPE\")\n        verify_two_t4_runtime(torch.cuda)\n        native = load_pinned_native_class(binding)\n        require(native.__name__ == \"EarlyPairedTrainer\", \"NATIVE_ARM\")\n        trainer = native(**binding[\"constructor\"],\n                         overrides=binding[\"overrides\"])\n        require(trainer.world_size == 2 and trainer.ddp is False,\n                \"TWO_RANK_NATIVE\")\n        if rank == 0:\n            target = Path(trainer.best)\n            require(not target.exists(), \"NATIVE_BEST_ALREADY_EXISTS\")\n            target.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copyfile(G4 / \"best.pt\", target)\n            require(sha256(target) == sha256(G4 / \"best.pt\"),\n                    \"BEST_COPY_SHA\")\n        dist.barrier()\n    finally:\n        if dist.is_initialized():\n            dist.destroy_process_group()\n\n    # Historical FRESH plan above authenticates the R0 input only.\n    # The actual optimizer/model/EMA/RNG state is always native RESUME.\n    trainer._ddp_b16_g4_identity = dict(identity)\n    trainer._ddp_b16_g4_root = str(G4.resolve(strict=True))\n    trainer._ddp_b16_native_state_request = {\n        \"schema\": STATE_SCHEMA, \"mode\": \"RESUME_FROM_DDP_B16\",\n        \"identity\": identity, \"cursor\": cursor,\n        \"checkpoint_root\": str(G4.resolve(strict=True)),\n        \"externally_pinned_manifest_sha256\": EXTERNAL_PIN,\n    }\n    trainer._ddp_b16_verified_worker_ready = True\n    trainer._ddp_b16_g4_native_bound = True\n    trainer._ddp_b16_terminal_candidate_ready = True\n\n    start = time.monotonic()\n    last_epoch_end = start\n    state = {\"manifest\": EXTERNAL_PIN, \"next_epoch\": cursor[\"next_epoch\"]}\n    old_hook = hook.ddp_b16_g4_epoch_end\n\n    def session_hook(obj, *, epoch, nb, last_opt_step):\n        nonlocal last_epoch_end\n        result = old_hook(obj, epoch=epoch, nb=nb, last_opt_step=last_opt_step)\n        if epoch == 99:\n            require(result.get(\"status\") ==\n                    \"TERMINAL_EPOCH_G4_CAPSULE_NOT_WRITTEN\",\n                    \"TERMINAL_STATUS\")\n            return result\n        require(result.get(\"status\") ==\n                \"LOCAL_SYNCHRONIZED_DDP_CAPSULE_EXTERNAL_AUTHENTICATION_PENDING\",\n                \"EPOCH_G4\")\n        require(epoch == state[\"next_epoch\"], \"EPOCH_SEQUENCE\")\n        consensus = [None, None]\n        dist.all_gather_object(consensus, {\n            \"rank\": rank,\n            \"manifest\": result[\"manifest_sha256\"],\n            \"best\": result[\"selected_best_sha256\"],\n        })\n        require([x[\"rank\"] for x in consensus] == [0, 1] and\n                len({x[\"manifest\"] for x in consensus}) == 1 and\n                len({x[\"best\"] for x in consensus}) == 1,\n                \"G4_RANK_CONSENSUS\")\n        message = [None]\n        if rank == 0:\n            try:\n                saved = persist_epoch(obj, result, identity, epoch,\n                                      state[\"manifest\"])\n                now = time.monotonic()\n                elapsed = now - start\n                this_epoch = now - last_epoch_end\n                stop = (elapsed >= BUDGET_HOURS * 3600 or\n                        elapsed + max(5400.0, 1.5 * this_epoch)\n                        >= 11.0 * 3600)\n                message[0] = {\n                    \"ok\": True, \"stop\": stop,\n                    \"manifest_sha256\": saved[\"manifest_sha256\"],\n                }\n            except Exception as exc:\n                message[0] = {\"ok\": False, \"error\": str(exc)[:300]}\n        dist.broadcast_object_list(message, src=0)\n        require(message[0][\"ok\"],\n                \"EXPORT_\" + str(message[0].get(\"error\", \"failed\")))\n        require(message[0][\"manifest_sha256\"] == result[\"manifest_sha256\"],\n                \"EXPORT_HASH_RANK\")\n        state[\"manifest\"] = result[\"manifest_sha256\"]\n        state[\"next_epoch\"] = epoch + 1\n        last_epoch_end = time.monotonic()\n        dist.barrier()\n        if message[0][\"stop\"]:\n            raise EpochBoundaryStop(\"SESSION_BUDGET_REACHED_AFTER_G4\")\n        return result\n\n    hook.ddp_b16_g4_epoch_end = session_hook\n    try:\n        safe = [np._core.multiarray.scalar, np.dtype]\n        safe.extend(type(np.dtype(n)) for n in (\n            \"float16\", \"float32\", \"float64\", \"int16\", \"int32\",\n            \"int64\", \"uint8\", \"uint16\", \"uint32\", \"uint64\", \"bool\",\n        ))\n        safe = list(dict.fromkeys(safe))\n        try:\n            with torch.serialization.safe_globals(safe):\n                trainer.train()\n            require(state[\"next_epoch\"] == 99 and trainer.epoch == 99,\n                    \"UNEXPECTED_NATIVE_STOP\")\n            status = \"EPOCH100_COMPLETED\"\n        except EpochBoundaryStop:\n            status = \"PAUSED_AT_G4\"\n\n        done = [None]\n        if rank == 0:\n            try:\n                terminal = persist_terminal(trainer, identity) if (\n                    status == \"EPOCH100_COMPLETED\") else None\n                done[0] = {\n                    \"ok\": True, \"status\": status,\n                    \"next_epoch\": 100 if terminal else state[\"next_epoch\"],\n                    \"manifest_sha256\": state[\"manifest\"],\n                    \"terminal\": terminal,\n                    \"training_executed\": True,\n                    \"b_test_access\": \"NONE\",\n                }\n            except Exception as exc:\n                done[0] = {\"ok\": False, \"error\": str(exc)[:300]}\n        dist.broadcast_object_list(done, src=0)\n        require(done[0][\"ok\"],\n                \"TERMINAL_\" + str(done[0].get(\"error\", \"unknown\")))\n        if rank == 0:\n            print(\"MV00_CONTINUOUS_RESULT=\" +\n                  json.dumps(done[0], sort_keys=True), flush=True)\n        dist.barrier()\n    finally:\n        hook.ddp_b16_g4_epoch_end = old_hook\n        if dist.is_initialized():\n            dist.destroy_process_group()\n\n\nif __name__ == \"__main__\":\n    main()\n"

# MV00 DDP-B16 CONTINUOUS TRAINING — Kaggle Save Version / Save & Run All.
# Pin the previous output's manifest SHA and mounted dataset root on each new version.
# NO B-TEST. Fixed science / two T4 / batch16 / imgsz1024 / seed42 / 100 epochs.
PREVIOUS_OUTPUT_DATASET = "/kaggle/input/datasets/awaishit/mv00-b16-acceptance-output"
PREVIOUS_MANIFEST_SHA256 = "4b8707bb9c4f16f42fa8796efd9777127e1cad9b9b5055a629e96b55607129b5"

from pathlib import Path
import hashlib, importlib.util, json, os, shutil, subprocess, sys
import torch

SOURCE_HEAD = "f440d329d4544ec08ad63c1972bcf9a8b8a7eed8"
BUNDLE_SHA256 = "4dc26b7277d58ed0b1fbcc1f4fac68aaf7ad417951e735bc8be2b040764861bb"
BUNDLE = Path("/kaggle/input/datasets/awaishit/resema-ddp-b16-acceptance/RESEMA_DDP_B16_ACCEPTANCE.bundle")
PRIOR = Path(PREVIOUS_OUTPUT_DATASET)
WORK = Path("/kaggle/working/MV00_B16_CONTINUOUS")
REPO, STAGE, G4 = WORK/"repo", WORK/"stage", WORK/"g4"
OUT = Path("/kaggle/working/MV00_B16_CONTINUOUS_OUTPUT")
RUN_ID = "MV00_B16_FRESH_ACCEPTANCE"
INPUTS = {
 "n1":Path("/kaggle/input/datasets/awaishit/detection-frozen-git-n1-accepted-r1"),
 "handoff":Path("/kaggle/input/datasets/awaishit/detection-project-training-handoff-r1"),
 "legacy":Path("/kaggle/input/datasets/awaishit/detection-legacy-notebook-files-flat-r1"),
 "split_b":Path("/kaggle/input/datasets/utopianstar/grazpedwri-dx-split-b"),
 "assets":Path("/kaggle/input/datasets/nithoukhan/d4e-r12-accepted-assets"),
 "archives":Path("/kaggle/input/datasets/nithoukhan/d4e-r12-raw-archives-v1"),
 "evidence":Path("/kaggle/input/datasets/nithoukhan2/d4f-r7d-legacy-exact-source-evidence-r1"),
}
def sha256(p):
 h=hashlib.sha256()
 with Path(p).open("rb") as f:
  for b in iter(lambda:f.read(4*1024*1024),b""):h.update(b)
 return h.hexdigest()

assert PRIOR.is_dir() and BUNDLE.is_file() and not BUNDLE.is_symlink()
assert all(p.is_dir() for p in INPUTS.values()), "MISSING_ORIGINAL_INPUT_DATASET"
assert not WORK.exists() and not OUT.exists(), "USE_CLEAN_KAGGLE_SESSION"
assert sha256(BUNDLE)==BUNDLE_SHA256, "SOURCE_BUNDLE_MISMATCH"
assert torch.cuda.is_available() and torch.cuda.device_count()==2
assert all("T4" in torch.cuda.get_device_name(i).upper() for i in (0,1))

receipts = sorted(p for p in PRIOR.rglob("ACCEPTANCE.json")
                  if p.is_file() and not p.is_symlink() and p.parent.name=="g4")
assert len(receipts)==1, "EXACTLY_ONE_PERSISTED_G4_REQUIRED"
receipt=json.loads(receipts[0].read_text("utf-8"))
src=receipts[0].parent
assert receipt["manifest_sha256"]==PREVIOUS_MANIFEST_SHA256, "EXTERNAL_CHECKPOINT_PIN_DRIFT"
assert receipt["run_id"]==RUN_ID and receipt["source_release_sha"]==SOURCE_HEAD
assert receipt["b_test_access"]=="NONE"
assert receipt["status"] in {
 "NONOFFICIAL_EPOCH1_G4_SAVED_NOT_OFFSESSION_VERIFIED",
 "MV00_RESUMABLE_EPOCH_G4_CANDIDATE"
}, "G4_RECEIPT_STATUS"
files=receipt["files_sha256"]
assert len(files)==6
assert all(Path(name).name==name and (src/name).is_file() and
           not (src/name).is_symlink() and sha256(src/name)==digest
           for name,digest in files.items()),"G4_MEMBER_SHA_FAILURE"
manifests=[name for name in files if name.endswith("_two_rank_manifest.json")]
assert len(manifests)==1
assert sha256(src/manifests[0])==PREVIOUS_MANIFEST_SHA256
manifest=json.loads((src/manifests[0]).read_text("utf-8"))
assert manifest["complete"] is True and set(manifest["ranks"])=={"0","1"}
assert manifest["identity"]["run_id"]==RUN_ID
assert manifest["identity"]["arm"]=="MV00"
assert manifest["identity"]["ddp_release_sha"]==SOURCE_HEAD
assert 1<=manifest["cursor"]["next_epoch"]<100

WORK.mkdir()
subprocess.run(["git","clone","--quiet","--branch","research/vgra-ddp-b16",
                str(BUNDLE),str(REPO)],check=True)
subprocess.run(["git","-C",str(REPO),"checkout","--quiet","--detach",SOURCE_HEAD],
               check=True)
assert subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],
                               text=True).strip()==SOURCE_HEAD
assert not subprocess.check_output(["git","-C",str(REPO),"status","--porcelain",
                                    "--untracked-files=no"],text=True).strip()
G4.mkdir()
for name,digest in files.items():
 shutil.copyfile(src/name,G4/name)
 assert sha256(G4/name)==digest

sys.dont_write_bytecode=True
sys.path.insert(0,str(REPO))
from research.r9_training.ddp_b16.r0_relocation import restore_r0_config
from research.r9_training.ddp_b16.ddp_worker import review_worker_inputs
from research.r9_training.ddp_b16.ddp_verified_admission import verify_staged_admission

config_files=sorted(INPUTS["n1"].rglob("R0_RELOCATED_CONFIG_CANDIDATE.json"))
assert len(config_files)==1
config=config_files[0]
r0_file=REPO/"research/r9_training/frozen_source/ultralytics/models/yolo/detect/d4e_r3_read_only_identity_preflight.py"
spec=importlib.util.spec_from_file_location("_ddp_b16_continuous_r0",r0_file)
r0=importlib.util.module_from_spec(spec)
spec.loader.exec_module(r0)
restore_r0_config(config,[INPUTS[k] for k in ("n1","handoff","legacy","assets","archives","evidence")],
 asset_pins=r0.ASSET_PINS,overlay_pins=r0.OVERLAY_PINS,code_pins=r0.R21_PINS,
 native_blob_pin=r0.NATIVE_BLOB_SHA1,
 runtime_yaml_pin="f2cc6e31f5f1be152f4dd7603a397e142963923f1d35de613eb7c61b168bf0c0",
 mirrored_handoff=INPUTS["handoff"],
 archives_roots=[INPUTS["assets"],INPUTS["archives"]])
plan={
 "schema":"VGRA_DDP_B16_WORKER_PLAN_REVIEW_V1","arm":"MV00",
 "run_id":RUN_ID,"mode":"FRESH_FROM_EARLY", # source review only; actual state RESUME
 "checkout_root":str(REPO),"expected_git_head":SOURCE_HEAD,
 "r0_input_config":str(config),"stage_root":str(STAGE),"b_test_access":"NONE"
}
plan_path=WORK/"plan.json"
plan_path.write_text(json.dumps(plan,sort_keys=True,indent=2)+"\n","utf-8")
review=review_worker_inputs(plan,stage_source=True)
admission=verify_staged_admission(plan,review)
assert admission["identity"]==manifest["identity"],"G4_SOURCE_ADMISSION_DRIFT"

OUT.mkdir()
worker_file=WORK/"multiepoch_worker.py"
worker_file.write_text(WORKER_CODE,encoding="utf-8")
env=dict(os.environ)
env.update({"CUDA_VISIBLE_DEVICES":"0,1","OMP_NUM_THREADS":"1",
            "PYTHONDONTWRITEBYTECODE":"1","PYTHONPATH":str(REPO)+os.pathsep+env.get("PYTHONPATH",""),
            "DDP_B16_PREVIOUS_MANIFEST_SHA256":PREVIOUS_MANIFEST_SHA256,
            "DDP_B16_SESSION_HOURS":"9.0"})
cmd=[sys.executable,"-m","torch.distributed.run","--standalone","--nnodes=1",
     "--nproc_per_node=2","--max_restarts=0",str(worker_file)]
print("MV00 B16 CONTINUOUS RUN START; RESUMING EPOCH",
      manifest["cursor"]["next_epoch"]+1,"OF 100",flush=True)
print("SOURCE",SOURCE_HEAD,"B-TEST SEALED",flush=True)
log=OUT/"MV00_CONTINUOUS_LOG.txt"
with log.open("x",encoding="utf-8") as file:
 process=subprocess.run(cmd,cwd=REPO,env=env,stdout=file,stderr=subprocess.STDOUT)
tail=log.read_text("utf-8")
results=[s.split("=",1)[1] for s in tail.splitlines()
         if s.startswith("MV00_CONTINUOUS_RESULT=")]
if process.returncode or len(results)!=1:
 print(tail[-12000:])
 raise RuntimeError("CONTINUOUS_RUN_FAILED_INSPECT_"+str(log))
result=json.loads(results[0])
assert result["b_test_access"]=="NONE" and result["training_executed"] is True
if result["status"]=="PAUSED_AT_G4":
 latest=OUT/"g4"/"ACCEPTANCE.json"
 assert latest.is_file()
 new=json.loads(latest.read_text("utf-8"))
 assert new["manifest_sha256"]==result["manifest_sha256"]
 assert new["next_epoch"]>manifest["cursor"]["next_epoch"]
 for name,digest in new["files_sha256"].items():
  assert sha256(OUT/"g4"/name)==digest
 print("SESSION PASS. NEXT EPOCH:",new["next_epoch"]+1,"/100")
 print("PERSISTED MANIFEST SHA:",new["manifest_sha256"])
 print("SAVE VERSION AND ATTACH ITS OUTPUT TO THE NEXT SESSION.")
else:
 assert result["status"]=="EPOCH100_COMPLETED"
 assert (OUT/"terminal"/"TERMINAL.json").is_file()
 print("EPOCH100 COMPLETED. TERMINAL BEST B-VAL OUTPUT READY.")
print("OUTPUT:",OUT,"B-TEST SEALED")
